In [1]:
# Cash Belknap
# cashbelknap@utexas.edu
# ctb2559

### Context
A large chunk of the code I've provided has been split into functions. Originally the code pieces were not in functions, and I just commented them out once I was done with them. However, since I am submitting this code I wanted to make it more readable but still runnable without having to wait 8 hours. That's why I opted to put everything into functions and left the calls commented out at the bottom of each block. Because I did this without too much thought, there may be some variables declared in one block that need to be accesible elsewhere but aren't. I don't think there should be, but if there is it worked fine when I ran it, and I just accidentally boxed it out when cleaning the file, so sorry if that's the case.

In [2]:
# Imports
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime
from pandas import DataFrame
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer

from sklearn.ensemble import VotingClassifier, StackingClassifier, RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier


from linear_regression_ensemble import LinearRegressionEnsemble

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.decomposition import PCA

In [3]:
# Enable inline mode for matplotlib so that IPython displays graphs.
%matplotlib inline

In [4]:
data = pd.read_csv("train.csv", skipinitialspace=True)
test = pd.read_csv("test.csv", skipinitialspace=True)
data.head()

,Id,Name,Intake Time,Found Location,Intake Type,Intake Condition,Animal Type,Sex upon Intake,Age upon Intake,Breed,Color,Outcome Time,Date of Birth,Outcome Type
0,A706918,Belle,07/05/2015 12:59:00 PM,9409 Bluegrass Dr in Austin (TX),Stray,Normal,Dog,Spayed Female,8 years,English Springer Spaniel,White/Liver,07/05/2015 03:13:00 PM,07/05/2007,Return to Owner
1,A724273,Runster,04/14/2016 06:43:00 PM,2818 Palomino Trail in Austin (TX),Stray,Normal,Dog,Intact Male,11 months,Basenji Mix,Sable/White,04/21/2016 05:17:00 PM,04/17/2015,Return to Owner
2,A857105,Johnny Ringo,05/12/2022 12:23:00 AM,4404 Sarasota Drive in Austin (TX),Public Assist,Normal,Cat,Neutered Male,2 years,Domestic Shorthair,Orange Tabby,05/12/2022 02:35:00 PM,05/12/2020,Transfer
3,A743852,Odin,02/18/2017 12:46:00 PM,Austin (TX),Owner Surrender,Normal,Dog,Neutered Male,2 years,Labrador Retriever Mix,Chocolate,02/21/2017 05:44:00 PM,02/18/2015,Return to Owner
4,A635072,Beowulf,04/16/2019 09:53:00 AM,415 East Mary Street in Austin (TX),Public Assist,Normal,Dog,Neutered Male,6 years,Great Dane Mix,Black,04/18/2019 01:45:00 PM,06/03/2012,Return to Owner


Starting off, lets get some unnecessary columns out of the way. Id and Name are useless in training, Outcome Time may cause leakage, and DOB is basically the same as age, so I removed all of those to start with.

In [6]:
labels = ['Id', 'Name', 'Outcome Time', 'Date of Birth']
data.drop(labels=labels, axis=1, inplace=True)
data.head()

,Intake Time,Found Location,Intake Type,Intake Condition,Animal Type,Sex upon Intake,Age upon Intake,Breed,Color,Outcome Type
0,07/05/2015 12:59:00 PM,9409 Bluegrass Dr in Austin (TX),Stray,Normal,Dog,Spayed Female,8 years,English Springer Spaniel,White/Liver,Return to Owner
1,04/14/2016 06:43:00 PM,2818 Palomino Trail in Austin (TX),Stray,Normal,Dog,Intact Male,11 months,Basenji Mix,Sable/White,Return to Owner
2,05/12/2022 12:23:00 AM,4404 Sarasota Drive in Austin (TX),Public Assist,Normal,Cat,Neutered Male,2 years,Domestic Shorthair,Orange Tabby,Transfer
3,02/18/2017 12:46:00 PM,Austin (TX),Owner Surrender,Normal,Dog,Neutered Male,2 years,Labrador Retriever Mix,Chocolate,Return to Owner
4,04/16/2019 09:53:00 AM,415 East Mary Street in Austin (TX),Public Assist,Normal,Dog,Neutered Male,6 years,Great Dane Mix,Black,Return to Owner


In [7]:
print(data.isnull().sum())

Intake Time         0
Found Location      0
Intake Type         0
Intake Condition    0
Animal Type         0
Sex upon Intake     2
Age upon Intake     1
Breed               0
Color               0
Outcome Type        0
dtype: int64


Then there's some `null` data. Since there's only 3 null values in the whole data set, it's easiest to just drop those records.

So a lot of the following cleaning I hadn't done before the `LinearRegressionEnsemble`, which I believe contributed to lower accuracy on this model, since a lot of these values would originally be very unique from each other (like the Intake Time), leading to not being able to find a correlation. After (failing to do) hyperparameter tuning on the LRE, I realized I needed to do more cleaning.

In [9]:
data.dropna(inplace=True)

intake_time = pd.to_datetime(data['Intake Time'], format='%m/%d/%Y %I:%M:%S %p')

data.drop(labels=['Intake Time'], axis=1, inplace=True)

# Create new columns directly
data['Intake Day'] = intake_time.dt.day
data['Intake Month'] = intake_time.dt.month
data['Intake Year'] = intake_time.dt.year
data['Intake Time'] = intake_time.dt.hour

def age_to_days(age_str):
    if pd.isna(age_str):  
        return None
    num, unit = age_str.split()
    num = int(num)
    if 'year' in unit:
        return num * 365
    elif 'month' in unit:
        return num * 30
    elif 'week' in unit:
        return num * 7
    elif 'day' in unit:
        return num
    else:
        return 0

data['Age upon Intake'] = data['Age upon Intake'].apply(age_to_days)

def clean_location(loc_str):
    loc = loc_str.split()
    if len(loc) >= 2:
        return loc[-2] + loc[-1]
    else:
        return None

data['Found Location'] = data['Found Location'].apply(clean_location)

print(data.head())

  Found Location      Intake Type Intake Condition Animal Type  \
0     Austin(TX)            Stray           Normal         Dog   
1     Austin(TX)            Stray           Normal         Dog   
2     Austin(TX)    Public Assist           Normal         Cat   
3     Austin(TX)  Owner Surrender           Normal         Dog   
4     Austin(TX)    Public Assist           Normal         Dog   

  Sex upon Intake  Age upon Intake                     Breed         Color  \
0   Spayed Female             2920  English Springer Spaniel   White/Liver   
1     Intact Male              330               Basenji Mix   Sable/White   
2   Neutered Male              730        Domestic Shorthair  Orange Tabby   
3   Neutered Male              730    Labrador Retriever Mix     Chocolate   
4   Neutered Male             2190            Great Dane Mix         Black   

      Outcome Type  Intake Day  Intake Month  Intake Year  Intake Time  
0  Return to Owner           5             7         2015    

In [10]:
print(data.isnull().sum())

Found Location      0
Intake Type         0
Intake Condition    0
Animal Type         0
Sex upon Intake     0
Age upon Intake     0
Breed               0
Color               0
Outcome Type        0
Intake Day          0
Intake Month        0
Intake Year         0
Intake Time         0
dtype: int64


My reasons for the cleaning I did are as follows.
For the rest of the cleaning I opted to split the Intake Time into many fields because if it is left as is, there won't be any correlation between data points, as all dates and times will end up being unique. This is why I opted to split out day, month, year, and hour, as those values would provide more insight than the original.
As well, I changes Age Upon Intake to be the same units for all fields, being days, as it will be easier to interpret that way.
Finally, I changed the Found Location to just be the City/county they were found in, as including all the street addresses is much more difficult to deal with, unless you encode it as coorinate data, which I don't want to do.

### Approach:
    The plan is simple. Make a bunch of homogenous ensembles, and ensemble them together as a heterogeneous ensemble. 
    I'll make a Random Forest and Decision Stump ensemble, and for the rest of the approaches I'll determine whether or not it would make sense to ensemble them. If it does, I'll make an ensemble of them. If not I'll make a single model that will be added to the final ensemble. Also, any models that fall under a certain accurracy I'll cut. If it's an interior ensemble, I'll probably have the cutoff around 60-65% and if it's an interior model, but not an ensemble, I'll have the cutoff around 90%. The hope is that this will increase the total accuracy.
    As well, I want to try a lot of approaches because I think it'll be fun.

In [13]:
X = data.drop('Outcome Type', axis=1)
y = data['Outcome Type']

#### Starting with the fun thing
My first idea was making a Linear Regression ensemble. To do this I want to take 2 random features and have them split on subsets of the classifiers. For example one will be `Found Location` and `Intake Condition`, and it'll classify as either `[Died, Adoption]` or `[Return to Owner, Euthanasia, Transfer]`. I think it'll be an interesting approach, and thankfully ChatGPT understands what I'm trying to do, so getting the code to try this isn't terrible difficult. 

Instead of writing all the code in this file, I split it off into a seperate python file, `linear_regression_ensemble` 

Running the LRE with the defaults gets a 49.5% accuracy, which is better than guessing, but is under my target of 60%, so into hyperparameter tuning I go!

In [15]:
def train_lre():
    lr_X = X.copy()
    lr_y = y.copy()
    # print("X", lr_X.head())
    # print("\n y",lr_y.head())

    X_train, X_test, y_train, y_test = train_test_split(lr_X, lr_y, stratify=y)


    param_grid = {
        'base_estimator': [LinearRegression(), RandomForestClassifier(), DecisionTreeClassifier()],
        'n_models': [10,20],
        'features_per_model': [2,4]
    }
    gs = GridSearchCV(LinearRegressionEnsemble(), param_grid, cv=5)
    gs.fit(X_train, y_train)
    best_rf = gs.best_estimator_

    print(best_rf.get_params())

# train_lre()

Initially I had a lot more parameters in the models and features per model, but then it was running for 5+ hours, so I took it down a notch.
After running the GridSearchCV with these parameters it found that `RandomForestClassifier()`, `n_models=20`, and `features_per_model=4` were the best estimator parameters. I think that accuracy will increase as I increase n_models, but if I increase it too much it may fall in to overfitting, so I think I will run the classifier with `n_models=20` and another with `n_models=30`, and whichever has the best accuracy I'll take.

In [17]:
def manual_lre_train():
    lre = LinearRegressionEnsemble(base_estimator=RandomForestClassifier(), n_models=35, features_per_model=12)
    lre.fit(X_train, y_train)
    y_pred = lre.predict(X_test)
    print("Accuracy:", accuracy_score(y_test, y_pred))

    lre = LinearRegressionEnsemble(base_estimator=RandomForestClassifier(), n_models=30, features_per_model=10)
    lre.fit(X_train, y_train)
    y_pred = lre.predict(X_test)
    print("Accuracy:", accuracy_score(y_test, y_pred))

# manual_lre_train()

After messing around with the parameters more manually on my own, I found that `n_models=35` and `features_per_model=12` seems to get the best result, with around 64%, whereas `features_per_model=10` gets around 61%. Having less `features_per_model` decreases the accuracy, and having it `4` gets around 50%. Since I was able to get this model to over 60% I will add it to the ensemble, but it definitely isn't the same as my original intent for the model.

#### On to the Next Thing
With that amalgamate done, I'll move on to built in functions. The three single calssifiers I'll be adding next are KNN, Naive Bayes, and SVM.
This (should) be a straightforward process of testing each one and hyper parameter tuning, which will take some time but shouldn't be as crazy as the previous model.

Firstly though, I need to normalize the data for these, so thanks ChatGPT for the function to do so.

In [20]:
def normalize_df(df: pd.DataFrame, preprocessor: ColumnTransformer = None, fit: bool = True):
    """
    Normalize and one-hot encode a DataFrame using a consistent preprocessing pipeline.
    
    Parameters:
        df (pd.DataFrame): Input DataFrame.
        preprocessor (ColumnTransformer, optional): If provided, uses this fitted preprocessor.
        fit (bool): If True, fits the preprocessor to the data; otherwise, transforms only.
    
    Returns:
        normalized_df (pd.DataFrame): The transformed DataFrame.
        preprocessor (ColumnTransformer): The fitted preprocessor (if fit=True), else the original one.
        feature_names (List[str]): The final column names after transformation.
    """

    # Identify column types
    numeric_features = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
    categorical_features = df.select_dtypes(include=['object', 'category']).columns.tolist()

    if fit:
        # Define transformers
        numeric_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='mean')),
            ('scaler', StandardScaler())
        ])

        categorical_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=True))
        ])

        # Create new preprocessor
        preprocessor = ColumnTransformer(
            transformers=[
                ('num', numeric_transformer, numeric_features),
                ('cat', categorical_transformer, categorical_features)
            ]
        )

        processed = preprocessor.fit_transform(df)
    else:
        if preprocessor is None:
            raise ValueError("Must provide a fitted preprocessor if fit=False.")
        processed = preprocessor.transform(df)

    # Get feature names
    encoded_cat_names = preprocessor.named_transformers_['cat'].named_steps['encoder'].get_feature_names_out(categorical_features)
    feature_names = numeric_features + list(encoded_cat_names)

    # Convert to DataFrame
    normalized_df = pd.DataFrame(
        processed.toarray() if hasattr(processed, 'toarray') else processed,
        columns=feature_names,
        index=df.index
    )

    return normalized_df, preprocessor, feature_names


In [21]:
def train_knn():
    print("Beginning KNN")
    kn_X = X.copy()
    kn_y = y.copy()

    kn_X, _, _ = normalize_df(kn_X)

    # The program began erroring out about not having enough memory w/o PCA
    pca = PCA(n_components=0.95)
    kn_X = pca.fit_transform(kn_X)

    assert not np.isnan(kn_X).any(), "NaN found in features"
    assert np.isfinite(kn_X).all(), "Infinite values found in features"

    knn = KNeighborsClassifier()
    param_grid = {
        'n_neighbors': [9, 11, 13, 15],
        'weights': ['uniform', 'distance'],
        'p': [1, 2]
    }

    gs = GridSearchCV(knn, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
    gs.fit(kn_X, kn_y)

    best_knn = gs.best_estimator_
    print("Best KNN Params:", gs.best_params_)

    cv = cross_val_score(best_knn, kn_X, kn_y, cv=5, scoring='accuracy')
    print("KNN CV Accuracy Scores:", cv)
    print("KNN Mean Accuracy:", cv.mean())

# train_knn()

Initially n_neighbors was \[3, 5, 7, 9], which got 59% with {'n_neighbors': 9, 'p': 2, 'weights': 'uniform'}, so I increased the number of neighbors.

That ended me with this as the final output
Best KNN Params: {'n_neighbors': 15, 'p': 2, 'weights': 'uniform'}
KNN CV Accuracy Scores: \[0.60141244 0.59992803 0.59853358 0.60096262 0.60654042]
KNN Mean Accuracy: 0.6014754172102019

In [23]:
def train_nb():
    print("Beginning Naive Bayes")
    nb_X = X.copy()
    nb_y = y.copy()

    nb_X, _ = normalize_df(nb_X)

    # pca = PCA(n_components=0.95)
    # nb_X = pca.fit_transform(nb_X)

    nb = GaussianNB()
    param_grid = {
        'var_smoothing': [1e-3, 1e-1, 0.5, 1e1, 1e3]
    }

    gs = GridSearchCV(nb, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
    gs.fit(nb_X, nb_y)

    best_nb = gs.best_estimator_
    print("Best NB Parameters:", gs.best_params_)

    # Evaluate using cross_val_score
    cv = cross_val_score(best_nb, nb_X, nb_y, cv=5, scoring='accuracy')
    print("NB CV Accuracy Scores:", cv)
    print("NB Mean Accuracy:", cv.mean())

# train_nb()

With 'var_smoothing': \[1e-11, 1e-9, 1e-7, 1e-5, 1e-3], 1e-3 was selected with a 25% accuracy, so I tested with higher smoothing

That led to
Beginning Naive Bayes
Best NB Parameters: {'var_smoothing': 0.1}
NB CV Accuracy Scores: \[0.57032972 0.56817057 0.56502182 0.57172417 0.56493185]
NB Mean Accuracy: 0.5680356259277586

Not as high as I would like, but close enough I guess.

In [25]:
def train_svm():
    print("Beginning SVM")
    svm_X = X.copy()
    svm_y = y.copy()

    svm_X, _ = normalize_df(svm_X)

    pca = PCA(n_components=0.95)
    svm_X = pca.fit_transform(svm_X)

    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LinearSVC(max_iter=10000))  # Ensure convergence
    ])

    param_grid = {
        'clf__C': [0.01, 0.1, 1, 10, 100],
        'clf__loss': ['hinge', 'squared_hinge'],
        'clf__dual': [False, True]
    }

    gs = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
    gs.fit(svm_X, svm_y)

    best_linear_svc = gs.best_estimator_
    print("Best Parameters:", gs.best_params_)

    cv_scores = cross_val_score(best_linear_svc, svm_X, svm_y, cv=5, scoring='accuracy')
    print("SVM CV Accuracy Scores:", cv_scores)
    print("SVM Mean Accuracy:", cv_scores.mean())

# train_svm()

This one ended with 
Best Parameters: {'clf__C': 0.1, 'clf__dual': True, 'clf__loss': 'squared_hinge'}
SVM CV Accuracy Scores: \[0.58643336 0.58454411 0.59025685 0.59304575 0.59318069]
SVM Mean Accuracy: 0.5894921506005129

##### Initial Thoughts
Well, they took a long time to all run, but they all got around 60%, which is good enough that I'll throw them in.
So time to drop in a Neural Net

In [27]:
def train_nn():
    print("Beginning Neural Network")

    # Prepare the data
    nn_X = X.copy()
    nn_y = y.copy()
    
    # Normalize (and encode if needed)
    nn_X, _ = normalize_df(nn_X)

    pca = PCA(n_components=0.95)
    nn_X = pca.fit_transform(nn_X)

    # Define the model
    mlp = MLPClassifier(max_iter=1000, random_state=42)

    # Define hyperparameter grid
    param_grid = {
        'hidden_layer_sizes': [(50,), (100,), (100, 50)],
        'activation': ['relu', 'tanh'],
        'alpha': [0.0001, 0.001, 0.01],  # L2 penalty
        'learning_rate': ['constant', 'adaptive']
    }

    # Grid search with cross-validation
    gs = GridSearchCV(mlp, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
    gs.fit(nn_X, nn_y)

    # Best estimator
    best_nn = gs.best_estimator_
    print("Best Neural Net Parameters:", gs.best_params_)

    # Cross-validation accuracy
    cv = cross_val_score(best_nn, nn_X, nn_y, cv=5, scoring='accuracy')
    print("NN CV Accuracy Scores:", cv)
    print("NN Mean Accuracy:", cv.mean())

# train_nn()

And another set of results
Best Neural Net Parameters: {'activation': 'tanh', 'alpha': 0.01, 'hidden_layer_sizes': (100,), 'learning_rate': 'constant'}
NN CV Accuracy Scores: \[0.60914939 0.60829472 0.61045387 0.61310782 0.61225316]
NN Mean Accuracy: 0.6106517925419459

Also, this is around when I started using AI to generate these code blocks. They're all basically the same, just swapping stuff out, so I got lazy.

##### The Frustration
Why does everything seem to be ~60% accuracy? I mean, it's good enough for the ensemble, but it'd be nice to see something in like 70% or something.

##### Continuing On
I'll do one last single model, a Decision Tree, then I'll move on to the internal ensembles.

In [30]:
def train_dt():
    print("Beginning Decision Tree")

    # Prepare the data
    dt_X = X.copy()
    dt_y = y.copy()

    # Normalize and encode if needed
    dt_X, _ = normalize_df(dt_X)

    # Dimensionality reduction
    # pca = PCA(n_components=0.95)
    # dt_X = pca.fit_transform(dt_X)

    # Define the classifier
    dt = DecisionTreeClassifier(random_state=42)

    # Hyperparameter grid
    param_grid = {
        'max_depth': [5, 10, 20, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'criterion': ['gini', 'entropy']  # or 'log_loss' for probabilistic output
    }

    # Grid search
    gs = GridSearchCV(dt, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
    gs.fit(dt_X, dt_y)

    # Best estimator
    best_dt = gs.best_estimator_
    print("Best Decision Tree Params:", gs.best_params_)

    # Evaluate using cross-validation
    cv_scores = cross_val_score(best_dt, dt_X, dt_y, cv=5, scoring='accuracy')
    print("Decision Tree CV Accuracy Scores:", cv_scores)
    print("Decision Tree Mean Accuracy:", cv_scores.mean())

# train_dt()

Best Decision Tree Params: {'criterion': 'gini', 'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 2}
Decision Tree CV Accuracy Scores: \[0.62394854 0.6210247  0.62646755 0.62948135 0.63132563]
Decision Tree Mean Accuracy: 0.6264495524267913

Yet another one in the 60s

##### Travel into the time sinks
Well, on to training the ensembles. Hopefully we can get higher accurracy on those.

In [33]:
def train_rf():
    print("Beginning Random Forest")

    # Prepare the data
    rf_X = X.copy()
    rf_y = y.copy()

    # Normalize and encode if needed
    rf_X, _ = normalize_df(rf_X)

    # pca = PCA(n_components=0.95)
    # rf_X = pca.fit_transform(rf_X)

    # Define the model
    rf = RandomForestClassifier(random_state=42)

    # Hyperparameter grid
    param_grid = {
        'n_estimators': [100, 200],
        'max_depth': [None, 10, 20],
        'min_samples_split': [2, 5],
        'min_samples_leaf': [1, 2],
        'bootstrap': [True, False]
    }

    # Grid search
    gs = GridSearchCV(rf, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
    gs.fit(rf_X, rf_y)

    # Best estimator
    best_rf = gs.best_estimator_
    print("Best Random Forest Params:", gs.best_params_)

    # Evaluate using cross-validation
    cv_scores = cross_val_score(best_rf, rf_X, rf_y, cv=5, scoring='accuracy')
    print("Random Forest CV Accuracy Scores:", cv_scores)
    print("Random Forest Mean Accuracy:", cv_scores.mean())

# train_rf()

Beginning Random Forest
Best Random Forest Params: {'bootstrap': False, 'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 200}
Random Forest CV Accuracy Scores: \[0.61148846 0.61166839 0.6139175  0.62008007 0.62286897]
Random Forest Mean Accuracy: 0.61600467815213

In [35]:
def train_ds():
    print("Beginning Decision Stump Ensemble (AdaBoost)")

    # Copy and normalize data
    ds_X = X.copy()
    ds_y = y.copy()
    ds_X, _ = normalize_df(ds_X)

    # pca = PCA(n_components=0.95)
    # ds_X = pca.fit_transform(ds_X)

    # Base estimator: decision stump
    stump = DecisionTreeClassifier(max_depth=1)

    # AdaBoost with decision stumps
    ada = AdaBoostClassifier(estimator=stump, random_state=42)
    
    # Hyperparameter grid
    param_grid = {
        'n_estimators': [50, 100, 200],
        'learning_rate': [0.01, 0.1, 1.0]
    }

    # Grid search with CV
    gs = GridSearchCV(ada, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
    gs.fit(ds_X, ds_y)

    # Best estimator
    best_ada = gs.best_estimator_
    print("Best AdaBoost Params:", gs.best_params_)

    # Evaluate performance
    cv_scores = cross_val_score(best_ada, ds_X, ds_y, cv=5, scoring='accuracy')
    print("AdaBoost CV Accuracy Scores:", cv_scores)
    print("AdaBoost Mean Accuracy:", cv_scores.mean())

# train_ds()

Best AdaBoost Params: {'learning_rate': 1.0, 'n_estimators': 100}
AdaBoost CV Accuracy Scores: \[0.59965814 0.59884845 0.60285187 0.60217714 0.60600063]
AdaBoost Mean Accuracy: 0.6019072466375782

#### Finishing Up
Yippee, it only took like 3 hours to run these ensembles, and hey, still in the 60% range. I feel like I'm cursed to never break 65% on any of these models.

##### Descent into madness
If this final ensemble isn't any good I'm crashing out.

Time to bring it all together and train everything, dump it into an ensemble, pray, export it, and get the final predictions from the test data.

In [38]:
def i_love_it_when_a_plan_comes_together():
    # Fit preprocessor on the training data
    _, normalize_transformer, _ = normalize_df(X, fit=True)

    def apply_normalization_only(X):
        return normalize_transformer.transform(X)


    # Save for inference later
    # joblib.dump(normalize_transformer, 'preprocessor.pkl')

    normalize_transformer_func = FunctionTransformer(apply_normalization_only, validate=False)

    def to_dense(X):
        return X.toarray()


    estimators = [
        ('lre', Pipeline([
            ('lre', LinearRegressionEnsemble(base_estimator=RandomForestClassifier(), n_models=35, features_per_model=12))
        ])),
        ('knn', Pipeline([
            ('normalize', normalize_transformer_func),
            ('to_dense', FunctionTransformer(to_dense, validate=False)),
            ('pca', PCA(n_components=100, svd_solver='auto', random_state=42)),
            ('knn', KNeighborsClassifier(n_neighbors=15, weights='uniform', p=2))
        ])),
        ('mlp', Pipeline([
            ('normalize', normalize_transformer_func),
            ('to_dense', FunctionTransformer(to_dense, validate=False)),
            ('pca', PCA(n_components=100, svd_solver='auto', random_state=42)),
            ('mlp', MLPClassifier(max_iter=1000, activation='tanh', alpha=0.01, hidden_layer_sizes=(100,), learning_rate='constant'))
        ])),

        ('nb', Pipeline([
            ('normalize', normalize_transformer_func),
            ('to_dense', FunctionTransformer(to_dense, validate=False)),
            ('nb', GaussianNB(var_smoothing=0.1))
        ])),
        ('dt', Pipeline([
            ('normalize', normalize_transformer_func),
            ('dt', DecisionTreeClassifier(criterion='gini', max_depth=10, min_samples_leaf=1, min_samples_split=2))
        ])),
        ('rf', Pipeline([
            ('normalize', normalize_transformer_func),
            ('rf', RandomForestClassifier(bootstrap=False, max_depth=None, min_samples_leaf=2, min_samples_split=2, n_estimators=200))
        ])),
        ('ada', Pipeline([
            ('normalize', normalize_transformer_func),
            ('to_dense', FunctionTransformer(to_dense, validate=False)),
            ('ada', AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1), n_estimators=100))
        ]))
    ]

    # Assemble all into a VotingClassifier
    final_ensemble = VotingClassifier(
        estimators=estimators,
        voting='soft',
        n_jobs=1
    )

    # Fit the ensemble
    final_ensemble.fit(X, y)

    # Export the final model
    joblib.dump(final_ensemble, 'final_ensemble_model.joblib')
    print("✅ Final ensemble model trained and exported to 'final_ensemble_model.joblib'")

# i_love_it_when_a_plan_comes_together()

HAHAHAHAH IT COMPLIED AND DIDNT TAKE 60 YEARS

Okay. Time to then load in the model, clean the training data, then output the predictions. (Also, check the accurracy to see how cooked I am)

* Update. It didn't work because of one-hot encoding problems, so I made a change to the normalizer and hopefully it works now
* Update 2. It is not working. I think I might be a little cooked.
* Update 3. After a lot of debugging it works now. Basically I needed handle_unknown='ignore' in my OneHotEncoder in the normalize function.

Also, I would have like to use a stacking classifier, but I started to run out of reasonable time I could finish the model in, and I wanted to at least get a set of predictions, even if it wasn't the best possible, so I settled for a soft voting classifier.

Also also, I need to get the test data into the same format as the training data, so cleaning time!

In [42]:
id_col = test['Id']

intake_time = pd.to_datetime(test['Intake Time'], format='%m/%d/%y %H:%M')

test.drop(labels=['Id','Intake Time', 'Date of Birth'], axis=1, inplace=True)

# Create new columns directly
test['Intake Day'] = intake_time.dt.day
test['Intake Month'] = intake_time.dt.month
test['Intake Year'] = intake_time.dt.year
test['Intake Time'] = intake_time.dt.hour

test['Age upon Intake'] = test['Age upon Intake'].apply(age_to_days)

test['Found Location'] = test['Found Location'].apply(clean_location)

# To Check they look similar
# print(test.head())
# print(data.head())

In [43]:
def let_it_end():
    # Read in the model, pass in the test, and export
    model = joblib.load('final_ensemble_model.joblib')

    pred = model.predict(test)

    results = pd.DataFrame({
        'Id': id_col,
        'Outcome': pred
    })

    results.to_csv('predictions.csv', index=False)
    print("Predictions exported")

# let_it_end()

Out of curiosity, let's test how just the LRE model does in prediction

In [45]:
def for_the_funny():
    lre = LinearRegressionEnsemble(base_estimator=RandomForestClassifier(), n_models=35, features_per_model=12)
    lre.fit(X, y)

    pred = lre.predict(test)

    results = pd.DataFrame({
        'Id': id_col,
        'Outcome': pred
    })

    results.to_csv('lre_predictions.csv', index=False)
    print("LRE Predictions exported")

# for_the_funny()

It's surprisingly close to the ensemble. Only a difference of .2

I think the fact that they're so close kind of hurts me inside, but also hey, I had fun doing a bunch of random stuff (except when I was dying from waiting for hours for it to train).